# Telco Customer Churn — dataset exploration

Milestone 2 notebook. The goal is to understand the raw IBM Telco Customer Churn file and record the cleaning decisions implemented in `src/preprocessing.py`. The Random Forest is **not** trained here.

Raw file: `data/raw/Telco-Customer-Churn.csv` (left unchanged).

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_raw_dataframe
from src.preprocessing import (
    CATEGORICAL_FEATURES,
    IDENTIFIER_COLUMNS,
    NUMERICAL_FEATURES,
    TARGET_COLUMN,
    clean_raw_dataframe,
)

raw = load_raw_dataframe()
raw.shape

## Schema and dtypes

`TotalCharges` is stored as text even though it represents a dollar amount. `SeniorCitizen` is an integer flag (`0`/`1`), not a continuous measurement. `customerID` uniquely identifies a row.

In [ ]:
print("rows:", len(raw), "columns:", raw.shape[1])
print("column names:", list(raw.columns))
display(raw.dtypes.to_frame("dtype"))
display(raw.nunique().to_frame("n_unique"))

## Missing values and duplicates

pandas `isna()` is zero on the raw file. The real quality issue is **blank `TotalCharges` strings**, which only appear after numeric conversion.

In [ ]:
missing = pd.DataFrame({
    "missing_count": raw.isna().sum(),
    "missing_pct": (raw.isna().mean() * 100).round(4),
})
display(missing)

blank_total = raw["TotalCharges"].astype(str).str.strip().eq("").sum()
print("blank TotalCharges strings:", blank_total)
print("exact duplicate rows:", int(raw.duplicated().sum()))
print("duplicate customerID:", int(raw["customerID"].duplicated().sum()))

## Target distribution

Churn is imbalanced (about 73.5% No / 26.5% Yes). Class balancing is **not** applied in this milestone.

In [ ]:
counts = raw["Churn"].value_counts(dropna=False)
display(pd.DataFrame({
    "count": counts,
    "proportion": raw["Churn"].value_counts(normalize=True),
}))

## TotalCharges: text that must become numeric

Eleven customers have `tenure == 0` and a blank `TotalCharges`. All eleven have `Churn = No`. They are new customers with no accrued total, so cleaning sets those eleven values to `0` rather than dropping the rows. No other blanks appear.

In [ ]:
total_numeric = pd.to_numeric(raw["TotalCharges"], errors="coerce")
problem = raw.loc[total_numeric.isna(), ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]
print("rows that fail numeric conversion:", len(problem))
display(problem)
print("tenure == 0 count:", int((raw["tenure"] == 0).sum()))

## Numerical summaries

In [ ]:
display(raw[["tenure", "MonthlyCharges"]].describe())
display(total_numeric.describe().to_frame("TotalCharges (coerced)"))
print("negative tenure:", int((raw["tenure"] < 0).sum()))
print("negative MonthlyCharges:", int((raw["MonthlyCharges"] < 0).sum()))

## Categorical distributions

In [ ]:
cat_cols = [c for c in raw.columns if c not in ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]
for col in cat_cols:
    print(col, raw[col].value_counts(dropna=False).to_dict())

## Feature groups used by the pipeline

- **Target:** `Churn` (`No` to 0, `Yes` to 1)
- **Identifier:** `customerID` (not a model feature)
- **Numerical:** `tenure`, `MonthlyCharges`, `TotalCharges`
- **Categorical:** remaining attributes, including `SeniorCitizen` as a flag

Cleaning does not drop rows. The sklearn `ColumnTransformer` is built unfitted so later training can fit it on the training split only.

In [ ]:
cleaned = clean_raw_dataframe(raw)
print("identifier:", IDENTIFIER_COLUMNS)
print("numerical:", NUMERICAL_FEATURES)
print("categorical:", CATEGORICAL_FEATURES)
print("target:", TARGET_COLUMN)
print("cleaned rows:", len(cleaned), "raw rows:", len(raw))
print("encoded target counts:", cleaned[TARGET_COLUMN].value_counts().sort_index().to_dict())
print("TotalCharges remaining NA:", int(cleaned["TotalCharges"].isna().sum()))

## Key observations

1. 7,043 rows by 21 columns; unique `customerID`; no exact duplicate rows.
2. Target is binary and imbalanced (about 26.5% churn).
3. `TotalCharges` is the only systematic quality defect: 11 blanks at tenure 0, filled with 0.
4. Service fields use No internet service and No phone service as valid levels, not missing data. They stay as categories.
5. No outlier clipping and no feature selection in this milestone.
6. One-hot encoding and imputation belong in the unfitted sklearn pipeline, fitted later on training data only.